# GPS Tracker (TON_IoT) — Detection Pipeline
مواصفات الجروب (النسخة المعدّلة): Epoch 2019-01-01 · نافذة 10 ثواني · `date+time` · chunks · Dedup قبل الـ Split · Hash split (md5 + SALT) بـ BW=100 ونسب 70/15/15 · Purge أول 3 وآخر 3 نوافذ · اختيار الموديل والـ Threshold بالـ Validation فقط · Test مرة واحدة في الآخر.

In [ ]:
!pip install xgboost lightgbm catboost -q
import os, json, hashlib, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, precision_recall_curve

SEED, SOURCE = 42, "gps"
SENSORS = ["latitude", "longitude"]
EPOCH, WINDOW_SEC = "2019-01-01", 10
BLOCK, SALT, TRAIN_PCT, VAL_PCT = 100, "iot-anomaly", 70, 15
DATA_PATH = "Train_Test_IoT_GPS_Tracker.csv"
if not os.path.exists(DATA_PATH):
    from google.colab import files
    DATA_PATH = list(files.upload().keys())[0]

## 1) Dedup أولًا، ثم الوقت من date+time، chunks، نوافذ 10 ثواني

In [ ]:
raw = pd.read_csv(DATA_PATH)
raw["time"] = raw["time"].str.strip()
df = raw.drop_duplicates().reset_index(drop=True)                      # Dedup قبل أي حاجة
print("Rows:", len(raw), "->", len(df))

t = pd.to_datetime(df["date"] + " " + df["time"], format="%d-%b-%y %H:%M:%S")   # date+time (مش ts)
df["chunk"] = (t.diff().dt.total_seconds() < 0).cumsum()                         # chunk جديد لما الوقت يرجع لورا
df["win"] = ((t - pd.Timestamp(EPOCH)).dt.total_seconds() // WINDOW_SEC).astype(int)

g = df.groupby(["chunk", "win"], sort=False)
W = g[SENSORS].agg(["mean", "std", "min", "max"])
W.columns = [f"{a}_{b}" for a, b in W.columns]
W["n_records"] = g.size()
counts = pd.crosstab([df["chunk"], df["win"]], df["type"]).reindex(W.index)
W["type"] = counts.idxmax(axis=1).values                                # Majority vote
W["is_attack"] = (W["type"] != "normal").astype(int)                    # y_true
W = W.reset_index().fillna(0)
W["block"] = W["win"] // BLOCK                      # السياق (lags) جوه نفس الـ chunk ونفس الـ block بس

# سياق آخر 3 نوافذ: من الماضي فقط وجوه نفس الـ chunk
W["prev_n_records"] = W.groupby(["chunk", "block"])["n_records"].shift(1).fillna(0)
W["gap_prev"] = W.groupby(["chunk", "block"])["win"].diff().fillna(0)
for c in ["latitude_mean", "latitude_std", "longitude_mean", "longitude_std", "n_records"]:
    for k in (1, 2, 3):
        W[f"{c}_lag{k}"] = W.groupby(["chunk", "block"])[c].shift(k)
    W[f"{c}_roll3"] = W.groupby(["chunk", "block"])[c].transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
    W[f"{c}_chg"] = W[c] - W[f"{c}_roll3"]
W = W.fillna(0)

FEATURES = [c for c in W.columns if c not in ("chunk", "win", "block", "type", "is_attack")]   # مفيش type/label/date/time
print(f"Windows: {len(W)} | chunks: {W.chunk.nunique()} | Attack: {W.is_attack.mean():.1%} | Features: {len(FEATURES)}")
print(W["type"].value_counts().to_dict())

## 2) Split بالـ Hash (نفس دالة الليدر) + Purge

In [ ]:
def assign_split(win, bw=BLOCK, salt=SALT):
    block = np.asarray(win) // bw        # رقم البلوك المطلق من الـ epoch
    bucket = {b: int(hashlib.md5(f"{salt}:{b}".encode()).hexdigest(), 16) % 100 for b in np.unique(block)}
    r = np.array([bucket[b] for b in block])
    return np.where(r < TRAIN_PCT, "train", np.where(r < TRAIN_PCT + VAL_PCT, "val", "test"))

block_id = W["win"].values // BLOCK
split_name = assign_split(W["win"].values)
train, val, test = (np.where(split_name == s)[0] for s in ("train", "val", "test"))

pos = W["win"].values % BLOCK
edge = (pos < 3) | (pos >= BLOCK - 3)                    # Purge: أول 3 وآخر 3
train, val, test = [i[~edge[i]] for i in (train, val, test)]
assert not set(block_id[train]) & set(block_id[val]) and not set(block_id[train]) & set(block_id[test]) and not set(block_id[val]) & set(block_id[test])

X, y = W[FEATURES], W["is_attack"].values
y_tr, y_va, y_te = y[train], y[val], y[test]
print(pd.DataFrame({"windows": [len(train), len(val), len(test)], "attack %": [y_tr.mean()*100, y_va.mean()*100, y_te.mean()*100]},
                   index=["train", "val", "test"]).round(1))
cov = pd.DataFrame({"train": W["type"].iloc[train].value_counts(), "val": W["type"].iloc[val].value_counts(),
                    "test": W["type"].iloc[test].value_counts()}).fillna(0).astype(int)
print(cov)

## 3) Binary: الخمس موديلات (Early Stopping على Validation) + Threshold بالـ Validation

In [ ]:
spw = (y_tr == 0).sum() / max((y_tr == 1).sum(), 1)

def make_models():
    # الداتا صغيرة (آلاف النوافذ) فالأشجار ضحلة + min_samples_leaf/min_child_weight + Regularization لتجنب الحفظ
    m = {"DecisionTree": DecisionTreeClassifier(max_depth=8, min_samples_leaf=10, class_weight="balanced", random_state=SEED),
         "RandomForest": RandomForestClassifier(n_estimators=300, max_depth=10, min_samples_leaf=5, class_weight="balanced", n_jobs=-1, random_state=SEED)}
    try:
        from xgboost import XGBClassifier
        m["XGBoost"] = XGBClassifier(n_estimators=1000, learning_rate=0.05, max_depth=3, min_child_weight=5, subsample=0.8, colsample_bytree=0.8,
                                     reg_lambda=5, scale_pos_weight=spw, early_stopping_rounds=50, eval_metric="logloss", random_state=SEED, n_jobs=-1)
    except ImportError: print("xgboost مش متسطب")
    try:
        from lightgbm import LGBMClassifier
        m["LightGBM"] = LGBMClassifier(n_estimators=1000, learning_rate=0.05, num_leaves=8, min_child_samples=20, subsample=0.8, subsample_freq=1,
                                       colsample_bytree=0.8, reg_lambda=5, class_weight="balanced", random_state=SEED, verbose=-1)
    except ImportError: print("lightgbm مش متسطب")
    try:
        from catboost import CatBoostClassifier
        m["CatBoost"] = CatBoostClassifier(iterations=1000, learning_rate=0.05, depth=4, l2_leaf_reg=5, auto_class_weights="Balanced", random_seed=SEED, verbose=0)
    except ImportError: print("catboost مش متسطب")
    return m

def fit(name, m, Xtr, ytr, Xva, yva):
    if name == "XGBoost": m.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)
    elif name == "LightGBM":
        from lightgbm import early_stopping
        m.fit(Xtr, ytr, eval_set=[(Xva, yva)], callbacks=[early_stopping(50, verbose=False)])
    elif name == "CatBoost": m.fit(Xtr, ytr, eval_set=(Xva, yva), early_stopping_rounds=50, verbose=0)
    else: m.fit(Xtr, ytr)
    return m

def best_threshold(y_true, p):
    pr, rc, th = precision_recall_curve(y_true, p)
    f1 = 2 * pr * rc / (pr + rc + 1e-12)
    return float(th[np.argmax(f1[:-1])])

models, rows = {}, []
for name, m in make_models().items():
    m = fit(name, m, X.iloc[train], y_tr, X.iloc[val], y_va)
    p_va = m.predict_proba(X.iloc[val])[:, 1]
    th = best_threshold(y_va, p_va)
    models[name] = (m, th)
    p_tr = m.predict_proba(X.iloc[train])[:, 1]
    rows.append(dict(model=name, threshold=round(th, 3), train_f1=f1_score(y_tr, p_tr >= th), val_recall=recall_score(y_va, p_va >= th),
                     val_precision=precision_score(y_va, p_va >= th), val_f1=f1_score(y_va, p_va >= th)))
bin_results = pd.DataFrame(rows).set_index("model").round(3)
bin_results["gap_train_val"] = (bin_results["train_f1"] - bin_results["val_f1"]).round(3)   # فرق كبير = Overfitting
print(bin_results)                                        # Validation فقط: الـ Test متلمسش لحد الآخر
best = bin_results["val_f1"].idxmax()
th_final = models[best][1]
print("Best (by validation F1):", best, "| threshold:", round(th_final, 3))

## 4) Multi-class (نوع الهجوم)

In [ ]:
classes = sorted(W["type"].unique())
le = LabelEncoder().fit(classes)
y_m = le.transform(W["type"])
missing_train = [c for c in classes if c not in set(W["type"].iloc[train])]
print("أنواع مش موجودة في Train:", missing_train or "مفيش")

# الموديل بيتدرب على الأنواع الموجودة في Train بس، ونحوّل لترميز متصل
tr_classes = np.array(sorted(set(y_m[train])))
remap = {c: i for i, c in enumerate(tr_classes)}
inv = {i: c for c, i in remap.items()}
y_m_tr = np.array([remap[c] for c in y_m[train]])

def make_mc():
    m = {"DecisionTree": DecisionTreeClassifier(max_depth=8, min_samples_leaf=10, class_weight="balanced", random_state=SEED),
         "RandomForest": RandomForestClassifier(n_estimators=300, max_depth=10, min_samples_leaf=5, class_weight="balanced", n_jobs=-1, random_state=SEED)}
    try:
        from xgboost import XGBClassifier
        m["XGBoost"] = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=3, min_child_weight=3, reg_lambda=5, random_state=SEED, n_jobs=-1)
    except ImportError: pass
    try:
        from lightgbm import LGBMClassifier
        m["LightGBM"] = LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=8, class_weight="balanced", reg_lambda=5, random_state=SEED, verbose=-1)
    except ImportError: pass
    try:
        from catboost import CatBoostClassifier
        m["CatBoost"] = CatBoostClassifier(iterations=300, learning_rate=0.05, depth=4, auto_class_weights="Balanced", random_seed=SEED, verbose=0)
    except ImportError: pass
    return m

def mc_predict(m, Xs):
    p = np.asarray(m.predict(Xs)).ravel().astype(int)
    return np.array([inv[i] for i in p])

mc, rows = {}, []
for name, m in make_mc().items():
    m.fit(X.iloc[train], y_m_tr); mc[name] = m
    rows.append(dict(model=name, val_f1_macro=f1_score(y_m[val], mc_predict(m, X.iloc[val]), average="macro"),
                     val_f1_weighted=f1_score(y_m[val], mc_predict(m, X.iloc[val]), average="weighted")))
mc_results = pd.DataFrame(rows).set_index("model").round(3)
print(mc_results)
best_mc = mc_results["val_f1_macro"].idxmax()
print("Best multi-class (by validation macro-F1):", best_mc)

## 5) Anomaly Score: Z-score / Mahalanobis / Isolation Forest / Autoencoder (نفس إعدادات باقي الأجهزة)

In [ ]:
from sklearn.covariance import EmpiricalCovariance
norm_tr = train[y[train] == 0]                         # Normal من الـ Train بس

def to_percentile(raw):                                # Score خام -> 0..1 (مقارنة بنوافذ Normal في الـ Train)
    ref = np.sort(raw[norm_tr]); return np.searchsorted(ref, raw) / len(ref)

sc = StandardScaler().fit(X.iloc[norm_tr])
raw_z = np.abs(sc.transform(X)).mean(axis=1)                          # Z-score: mean(axis=1)
cov = EmpiricalCovariance().fit(X.iloc[norm_tr]); raw_m = cov.mahalanobis(X)   # Mahalanobis
iso = IsolationForest(n_estimators=200, random_state=SEED, n_jobs=-1).fit(X.iloc[norm_tr]); raw_i = -iso.score_samples(X)
sc_ae = StandardScaler().fit(X.iloc[norm_tr]); Xn = sc_ae.transform(X.iloc[norm_tr])
ae = MLPRegressor(hidden_layer_sizes=(32, 16, 8, 16, 32), activation="relu", solver="adam", max_iter=200,
                  early_stopping=True, n_iter_no_change=10, random_state=SEED).fit(Xn, Xn)
X_all = sc_ae.transform(X); raw_ae = ((X_all - ae.predict(X_all)) ** 2).mean(axis=1)
anom = {"Z-score": to_percentile(raw_z), "Mahalanobis": to_percentile(raw_m),
        "IsolationForest": to_percentile(raw_i), "Autoencoder": to_percentile(raw_ae)}

ths = np.linspace(0.01, 0.99, 99)
th_an = {k: ths[np.argmax([f1_score(y_va, v[val] >= t) for t in ths])] for k, v in anom.items()}   # threshold من الـ Validation
rows = []
for k, v in anom.items():
    for nm, ii in (("val", val), ("test", test)):
        pr = v[ii] >= th_an[k]; yy = y[ii]
        rows.append(dict(model=k, split=nm, threshold=round(th_an[k], 2),
                         auc=roc_auc_score(yy, v[ii]) if len(set(yy)) > 1 else np.nan,
                         recall=recall_score(yy, pr), precision=precision_score(yy, pr, zero_division=0), f1=f1_score(yy, pr)))
an_results = pd.DataFrame(rows).set_index(["model", "split"]).round(3)
print(an_results)
best_an = an_results.xs("val", level="split")["f1"].idxmax()          # الاختيار بـ val F1 (زي باقي الأجهزة)
print("Best anomaly method (by validation F1):", best_an)
print(pd.DataFrame({k: pd.Series(v).groupby(W["type"].values).mean() for k, v in anom.items()}).round(3))

## 5b) Ablation: من غير latitude_mean / longitude_mean (ممكن يبقوا Proxy للفترة الزمنية)

In [ ]:
drop = [c for c in FEATURES if c.startswith(("latitude_mean", "longitude_mean"))]
F2 = [c for c in FEATURES if c not in drop]
rows = []
for name, m in make_models().items():
    m = fit(name, m, X[F2].iloc[train], y_tr, X[F2].iloc[val], y_va)
    pv = m.predict_proba(X[F2].iloc[val])[:, 1]; th = best_threshold(y_va, pv)
    rows.append(dict(model=name, val_f1_without=f1_score(y_va, pv >= th), val_f1_with=bin_results.loc[name, "val_f1"]))
print(pd.DataFrame(rows).set_index("model").round(3))
print("لو الفرق صغير = الموديل مش معتمد على latitude_mean/longitude_mean كـ Proxy للوقت")

## 6) Calibration (على الـ Validation) + التقييم النهائي: Test مرة واحدة بس

In [ ]:
from sklearn.isotonic import IsotonicRegression
bm = models[best][0]
p_raw = bm.predict_proba(X)[:, 1]
calibrator = IsotonicRegression(out_of_bounds="clip").fit(p_raw[val], y_va)    # Calibration على الـ Validation بس
p_attack = calibrator.predict(p_raw)
th_final = best_threshold(y_va, p_attack[val])                                 # threshold على الاحتمال المعايَر
p_va, p_te = p_attack[val], p_attack[test]
val_f1, test_f1 = f1_score(y_va, p_va >= th_final), f1_score(y_te, p_te >= th_final)
print(f"FINAL  model={best}  threshold={th_final:.3f}  (calibrated)")
print(f"  val : recall={recall_score(y_va, p_va>=th_final):.3f} precision={precision_score(y_va, p_va>=th_final):.3f} F1={val_f1:.3f}")
print(f"  test: recall={recall_score(y_te, p_te>=th_final):.3f} precision={precision_score(y_te, p_te>=th_final):.3f} F1={test_f1:.3f}")
print("Test class distribution:", pd.Series(y_te).value_counts().to_dict(), "| types:", W["type"].iloc[test].value_counts().to_dict())
if len(set(y_te)) < 2:
    print("تحذير: الـ Test فيه Attack بس -> F1 و AUC على الـ Test مالهمش معنى. المرجع هو الـ Validation.")
pm = mc_predict(mc[best_mc], X.iloc[test])
print(f"  multi-class ({best_mc}) test macro-F1 = {f1_score(y_m[test], pm, average='macro'):.3f}")

## 7) التصدير (نفس schema باقي الأجهزة)

In [ ]:
os.makedirs("export", exist_ok=True)
split_col = np.full(len(W), "purged", dtype=object)
for idx, key in ((train, "train"), (val, "val"), (test, "test")): split_col[idx] = key
pred_type = le.inverse_transform(mc_predict(mc[best_mc], X))
is_att = p_attack >= th_final
attack_type = np.where(is_att, np.where(pred_type != "normal", pred_type, "unknown"), "none")

# predictions: p_attack معايَر، والـ Purge متشال من الملف
pred = pd.DataFrame({"source": SOURCE, "chunk": W["chunk"].values, "win": W["win"].values, "split": split_col,
                     "p_attack": p_attack, "y_true": W["is_attack"].values, "true_type": W["type"].values,
                     "pred_type": pred_type, "n_records": W["n_records"].values})
pred = pred[pred["split"] != "purged"].reset_index(drop=True)
pred.to_csv(f"export/{SOURCE}_predictions.csv", index=False)

# events: نفس أعمدة Weather بالترتيب
events = pd.DataFrame({"source": SOURCE,
                       "window_start": pd.Timestamp(EPOCH) + pd.to_timedelta(W["win"] * WINDOW_SEC, unit="s"),
                       "prediction": np.where(is_att, "Attack", "Normal"), "attack_prob": p_attack,
                       "attack_type": attack_type, "anomaly_score": anom[best_an],
                       "n_records": W["n_records"].values, "true_label": W["is_attack"].values, "split": split_col})
events.insert(0, "event_id", range(len(events)))
events.to_csv(f"export/{SOURCE}_events.csv", index=False)

cfg = {"source": SOURCE, "binary_model": best, "multiclass_model": best_mc, "calibrated": True,
       "threshold": float(th_final), "anomaly_method": best_an, "features": FEATURES, "classes": list(le.classes_),
       "window_seconds": WINDOW_SEC, "epoch": EPOCH, "block_windows": BLOCK,
       "split_rule": {"method": "md5 hash of block number", "salt": SALT, "train_pct": TRAIN_PCT, "val_pct": VAL_PCT},
       "val_f1": float(val_f1), "test_f1": float(test_f1)}
json.dump(cfg, open(f"export/{SOURCE}_config.json", "w"), indent=2)

print(list(pred.columns)); print(list(events.columns)); print(list(cfg.keys()))
print(pred["split"].value_counts().to_dict(), "| events:", events["split"].value_counts().to_dict())
import shutil; shutil.make_archive("GPS_export", "zip", "export")
try:
    from google.colab import files; files.download("GPS_export.zip")
except Exception: pass